Perfect — you want **production‑grade mastery challenges in Pydantic**, organized by difficulty levels, so you can practice, showcase them on GitHub, and impress recruiters. I’ll design them as **real‑world scenarios** (not toy problems), each phrased as a challenge or use case you’d face in production systems.  

Here’s the first batch (we’ll go topic by topic, pausing after each so you can confirm before I continue). Let’s start with **Topic 1: Model Definition & Validation**.

---

# 🧩 Topic 1: Model Definition & Validation

### 🎯 Very Easy (2–5)
- Define a `User` model with `username`, `age`, and `email`. Enforce `min_length` and `max_length` on `username`.
- Create a `Product` model where `price` must always be non‑negative.
- Add a default `created_at` field using `default_factory=datetime.utcnow`.

### 🎯 Easy (2–5)
- Build a `Customer` model where `email` must match a regex pattern.
- Enforce that `age` must be between 18 and 65.
- Add an alias so `user_id` can be provided as `uid` in JSON input.

### 🎯 Medium (2–5)
- Create a `Transaction` model where `amount` can be provided as either `str` or `float`, but must be coerced into `float`.
- Add a validator that ensures `discount` cannot exceed `amount`.
- Enforce that `currency` must be one of `USD`, `EUR`, `INR`.

### 🎯 Very Medium (2–5)
- Build a `Payment` model where `card_number` must pass Luhn algorithm validation.
- Ensure `expiry_date` is in the future.
- Add a validator that ensures `cvv` is exactly 3 digits.

### 🎯 Hard (2–5)
- Create a `WebhookEvent` model that validates HMAC signatures against a secret key.
- Ensure `payload` is valid JSON and matches a nested schema.
- Add a validator that rejects events older than 5 minutes (based on `timestamp`).

### 🎯 Very Hard (2–5)
- Build a `BankAccount` model where `balance` updates must be idempotent (same request twice doesn’t change state).
- Enforce that `account_number` must be unique across a simulated database.
- Add a validator that ensures withdrawals never exceed balance.

### 🎯 Extreme Hard (2–5)
- Create a `DistributedJob` model where `job_id` must be a UUID and `status` transitions must follow a finite state machine (`PENDING → RUNNING → COMPLETED/FAILED`).
- Validate that `retry_count` increments only when `status=FAILED`.
- Ensure `metadata` field can accept arbitrary JSON but must pass a schema check dynamically loaded from a file.

### 🎯 Very Extreme Hard (2–5)
- Build a `MultiTenantUser` model where:
  - `tenant_id` must exist in a simulated external service (mock API call).
  - `roles` must be validated against tenant‑specific role definitions.
  - Add cross‑field validation: if `role=admin`, enforce `mfa_enabled=True`.
- Create a `LedgerEntry` model where:
  - `entry_id` must be globally unique across multiple shards.
  - `amount` must balance against a double‑entry accounting system (debits = credits).
  - Validation must query a simulated external DB.

---

🔥 These are **production‑style challenges** recruiters love to see because they show you can handle **real constraints**: regex, HMAC, UUIDs, external APIs, state machines, and DB consistency.

👉 Do you want me to continue with **Topic 2: Nested Models & Relationships** next, or would you like to pause here and start implementing Topic 1 first?

In [ ]:
from pydantic import BaseModel, EmailStr, Field
from datetime import datetime

class User(BaseModel):
    username: str = Field(min_length = 3, max_length = 50)
    email: EmailStr
    age: int =  Field(gt = 0, lt = 120)

class Product(BaseModel):
    price: float = Field(ge = 0)
    created_at: datetime = Field(default_factory = datetime.utcnow)



In [3]:
from pydantic import BaseModel, Field
import re  
from uuid import uuid4 

class Customer(BaseModel):
    email: str = Field(pattern = r'^[\w\.-]+@[\w]+\.[\w]+$')
    user_id :int = Field(alias = 'uid')
    age: int = Field(ge = 18, le=65)





In [4]:
from typing import Union
from pydantic import Field, BaseModel, field_validator

class Transaction(BaseModel):
    amount: Union[float, str] = Field(ge = 0)
    discount: float = Field(ge = 0, le = 100) #this ensure the amount must be greater than or equal to discount
    currency: str

    @field_validator('currency')
    @classmethod
    def validate_currency(cls, value):
        value = value.upper()
        if value not in ('USD', 'EUR', 'INR'):
            raise ValueError("currency must be in USD, EUR, INR")
        return value

    
    



In [6]:
from pydantic import BaseModel, Field, field_validator
from datetime import datetime
import re

class Payment(BaseModel):
    card_number: str = Field(..., min_length=12, max_length=19)
    expiry_date: str  # format MM/YY
    cvv: str

    # Luhn algorithm for card validation
    @field_validator("card_number")
    @classmethod
    def validate_card_number(cls, value: str) -> str:
        if not value.isdigit():
            raise ValueError("Card number must contain only digits")
        if not cls.luhn_check(value):
            raise ValueError("Invalid credit card number (Luhn check failed)")
        return value

    @classmethod
    def luhn_check(cls, number: str) -> bool:
        total = 0
        reverse_digits = number[::-1]
        for i, digit in enumerate(reverse_digits):
            n = int(digit)
            if i % 2 == 1:
                n *= 2
                if n > 9:
                    n -= 9
            total += n
        return total % 10 == 0

    # Expiry date validation
    @field_validator("expiry_date")
    @classmethod
    def validate_expiry_date(cls, value: str) -> str:
        pattern = r"^(0[1-9]|1[0-2])\/([0-9]{2})$"
        if not re.match(pattern, value):
            raise ValueError("Expiry date must be in MM/YY format")
        
        month, year = value.split("/")
        exp_date = datetime(int("20" + year), int(month), 1)
        now = datetime.now()
        if exp_date < now.replace(day=1):
            raise ValueError("Card has expired")
        return value

    # CVV validation
    @field_validator("cvv")
    @classmethod
    def validate_cvv(cls, value: str) -> str:
        if not re.match(r"^\d{3}$", value):
            raise ValueError("CVV must be exactly 3 digits")
        return value


In [7]:
import hmac
import hashlib
import json
from datetime import datetime, timedelta
from pydantic import BaseModel, Field, field_validator

SECRET_KEY = b"supersecretkey"  # In production, load from env variable

class WebhookEvent(BaseModel):
    payload: dict
    signature: str
    timestamp: datetime = Field(...)

    @field_validator("payload")
    @classmethod
    def validate_payload(cls, value):
        # Ensure payload is JSON-serializable
        try:
            json.dumps(value)
        except Exception:
            raise ValueError("Payload must be valid JSON")
        return value

    @field_validator("signature")
    @classmethod
    def validate_signature(cls, value, values):
        payload = values.get("payload")
        if payload is None:
            raise ValueError("Payload must be validated first")

        # Compute HMAC SHA256
        computed_sig = hmac.new(
            SECRET_KEY,
            json.dumps(payload, separators=(",", ":")).encode("utf-8"),
            hashlib.sha256
        ).hexdigest()

        if not hmac.compare_digest(computed_sig, value):
            raise ValueError("Invalid signature")
        return value

    @field_validator("timestamp")
    @classmethod
    def validate_timestamp(cls, value):
        now = datetime.utcnow()
        if value < now - timedelta(minutes=5):
            raise ValueError("Event timestamp too old")
        return value


In [8]:
from pydantic import BaseModel, Field, field_validator

# let for example we extracted the info from db and stored in this variable
EXISTING_ACCOUNT_NUMBERS = {"1234567890"}

class BankAccount(BaseModel):
    account_number: str = Field(..., min_length=5)
    balance: float = Field(0.0, ge=0.0)

    @field_validator("account_number")
    @classmethod
    def check_unique_account(cls, value: str) -> str:
        if value in EXISTING_ACCOUNT_NUMBERS:
            raise ValueError("Account number already exists.")
        return value

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdrawal amount must be greater than zero.")
        if amount > self.balance:
            raise ValueError("Withdrawal exceeds balance.")
        
        self.balance -= amount


In [9]:
import json
import os
from uuid import UUID
from typing import Dict, Any, Literal
from pydantic import BaseModel, Field, field_validator, model_validator

# Simulated dynamic JSON schema file creation for validation
SCHEMA_FILE_PATH = "metadata_schema.json"
with open(SCHEMA_FILE_PATH, "w") as f:
    json.dump({"required_fields": ["worker_id", "region"]}, f)


class DistributedJob(BaseModel):
    job_id: UUID
    status: Literal["PENDING", "RUNNING", "COMPLETED", "FAILED"] = "PENDING"
    retry_count: int = Field(0, ge=0)
    metadata: Dict[str, Any]

    @model_validator(mode="before")
    @classmethod
    def validate_dynamic_metadata(cls, data: Any) -> Any:
        """Loads schema from JSON file dynamically and validates metadata keys."""
        if isinstance(data, dict) and "metadata" in data:
            if os.path.exists(SCHEMA_FILE_PATH):
                with open(SCHEMA_FILE_PATH, "r") as f:
                    schema = json.load(f)
                
                # Check for required fields defined in the JSON file
                required = schema.get("required_fields", [])
                provided = data["metadata"].keys()
                
                for field in required:
                    if field not in provided:
                        raise ValueError(f"Metadata missing required schema field: '{field}'")
        return data

    def transition_to(self, new_status: Literal["PENDING", "RUNNING", "COMPLETED", "FAILED"]) -> None:
        """Enforces state machine transitions and increments retry on FAILURE."""
        valid_transitions = {
            "PENDING": ["RUNNING"],
            "RUNNING": ["COMPLETED", "FAILED"],
            "COMPLETED": [],
            "FAILED": ["PENDING", "RUNNING"]  # Allowed transitions to retry the job
        }

        if new_status not in valid_transitions[self.status]:
            raise ValueError(f"Invalid transition from {self.status} to {new_status}")

        if new_status == "FAILED":
            self.retry_count += 1

        self.status = new_status


In [11]:
from typing import Any, Dict, List
from pydantic import BaseModel, Field , model_validator

# Simulated External Database for Validation
MOCK_EXTERNAL_TENANTS = {
    "tenant-abc": ["admin", "editor", "viewer"],
    "tenant-xyz": ["admin", "manager", "staff"],
}

class MultiTenantUser(BaseModel):
    tenant_id: str
    role: str
    mfa_enabled: bool = False

    @model_validator(mode="before")
    @classmethod
    def validate_tenant_and_role_exists(cls, data: Any) -> Any:
        """Runs BEFORE field parsing to check external API data constraints."""
        if not isinstance(data, dict):
            return data
            
        tenant_id = data.get("tenant_id")
        role = data.get("role")

        # 1. Critical check: tenant_id must exist
        if not tenant_id or tenant_id not in MOCK_EXTERNAL_TENANTS:
            raise ValueError(f"Tenant ID '{tenant_id}' does not exist in external system.")

        # 2. Critical check: role must exist within that specific tenant
        allowed_roles = MOCK_EXTERNAL_TENANTS[tenant_id]
        if role not in allowed_roles:
            raise ValueError(f"Role '{role}' is invalid for tenant '{tenant_id}'. Allowed: {allowed_roles}")

        return data

    @model_validator(mode="after")
    def enforce_admin_mfa(self) -> "MultiTenantUser":
        """Runs AFTER field parsing to enforce cross-field business logic."""
        if self.role == "admin" and not self.mfa_enabled:
            # Automatically upgrade to True instead of failing validation
            self.mfa_enabled = True
        return self


class LedgerEntry(BaseModel):
    # Field validation ensures no negative monetary amounts
    debits: float = Field(..., ge=0.0)
    credits: float = Field(..., ge=0.0)

    @model_validator(mode="after")
    def validate_double_entry(self) -> "LedgerEntry":
        """Ensures the accounting system balances perfectly."""
        if self.debits != self.credits:
            raise ValueError(f"Accounting mismatch: Debits ({self.debits}) must equal Credits ({self.credits}).")
        return self


Excellent — let’s move on to **Topic 2: Nested Models & Relationships** in the same structured way (difficulty ladder, production‑grade scenarios, use cases, and challenges).  

---

# 🧩 Topic 2: Nested Models & Relationships

### 🎯 Very Easy (2–5)
- Create a `Profile` model nested inside a `User` model (fields: `bio`, `website`).  
- Define a `Product` model with a nested `Category` model.  
- Ensure nested models validate automatically when passed as dicts.

---

### 🎯 Easy (2–5)
- Build an `Order` model containing a list of `Product` models.  
- Add a `Customer` model with a nested `Address` model (`street`, `city`, `zip`).  
- Validate that `zip` must be exactly 5 digits.

---

### 🎯 Medium (2–5)
- Create a `Company` model with a list of `Employee` models.  
- Ensure each `Employee` has a unique `employee_id`.  
- Add a validator that ensures at least one employee has `role="CEO"`.

---

### 🎯 Very Medium (2–5)
- Build a `School` model with nested `Classroom` models, each containing `Student` models.  
- Ensure each classroom has a maximum of 30 students.  
- Validate that each student has a unique `roll_number` within the classroom.

---

### 🎯 Hard (2–5)
- Create an `Invoice` model with nested `LineItem` models.  
- Ensure the sum of all line item totals equals the `invoice_total`.  
- Add a validator that rejects invoices with mismatched totals.

---

### 🎯 Very Hard (2–5)
- Build a `Project` model with nested `Task` models.  
- Enforce that tasks must follow dependencies (e.g., `task_b` cannot start until `task_a` is completed).  
- Add a validator that ensures no circular dependencies exist.

---

### 🎯 Extreme Hard (2–5)
- Create a `MicroserviceConfig` model with nested `Service` models.  
- Each service must declare its dependencies.  
- Validate that all dependencies exist and no cyclic dependency graph is formed.  
- Add a validator that ensures at least one service is marked as `entrypoint=True`.

---

### 🎯 Very Extreme Hard (2–5)
- Build a `SupplyChain` model with nested `Factory`, `Warehouse`, and `RetailStore` models.  
- Enforce cross‑model rules:
  - Factories must supply warehouses.  
  - Warehouses must supply stores.  
  - Inventory counts must balance across the chain.  
- Add a validator that ensures no warehouse or store has negative stock.  
- Bonus: Validate that total stock across all warehouses equals total stock across all stores (end‑to‑end consistency).

---


In [ ]:
from pydantic import BaseModel, Field

class Profile(BaseModel):
    bio: str
    website: str

class User(BaseModel):
    username: str
    profile: Profile

class Category(BaseModel):
    name: str

class Product(BaseModel):
    name: str
    category: Category


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Product(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    products: List[Product]

class Address(BaseModel):
    street: str
    city: str
    zip: str = Field(..., regex=r"^\d{5}$")

class Customer(BaseModel):
    name: str
    address: Address


In [ ]:
from pydantic import BaseModel, field_validator
from typing import List

class Employee(BaseModel):
    employee_id: str
    role: str

class Company(BaseModel):
    name: str
    employees: List[Employee]

    @field_validator("employees")
    @classmethod
    def validate_employees(cls, employees):
        ids = [e.employee_id for e in employees]
        if len(ids) != len(set(ids)):
            raise ValueError("Employee IDs must be unique")
        if not any(e.role == "CEO" for e in employees):
            raise ValueError("Company must have a CEO")
        return employees


In [ ]:
from pydantic import BaseModel, field_validator
from typing import List

class Student(BaseModel):
    roll_number: int
    name: str

class Classroom(BaseModel):
    students: List[Student]

    @field_validator("students")
    @classmethod
    def validate_students(cls, students):
        if len(students) > 30:
            raise ValueError("Classroom cannot exceed 30 students")
        roll_numbers = [s.roll_number for s in students]
        if len(roll_numbers) != len(set(roll_numbers)):
            raise ValueError("Duplicate roll numbers in classroom")
        return students

class School(BaseModel):
    name: str
    classrooms: List[Classroom]


In [ ]:
from pydantic import BaseModel, field_validator
from typing import List

class LineItem(BaseModel):
    description: str
    total: float

class Invoice(BaseModel):
    invoice_total: float
    line_items: List[LineItem]

    @field_validator("line_items")
    @classmethod
    def validate_invoice(cls, items, values):
        expected_total = sum(item.total for item in items)
        if "invoice_total" in values and values["invoice_total"] != expected_total:
            raise ValueError("Invoice total does not match sum of line items")
        return items


In [ ]:
from pydantic import BaseModel, field_validator
from typing import List

class Task(BaseModel):
    task_id: str
    depends_on: List[str] = []

class Project(BaseModel):
    tasks: List[Task]

    @field_validator("tasks")
    @classmethod
    def validate_dependencies(cls, tasks):
        task_ids = {t.task_id for t in tasks}
        for task in tasks:
            for dep in task.depends_on:
                if dep not in task_ids:
                    raise ValueError(f"Dependency {dep} not found")
        # Detect circular dependencies (simplified)
        visited = set()
        def dfs(task_id, stack):
            if task_id in stack:
                raise ValueError("Circular dependency detected")
            stack.add(task_id)
            for dep in next(t for t in tasks if t.task_id == task_id).depends_on:
                dfs(dep, stack)
            stack.remove(task_id)
        for t in tasks:
            dfs(t.task_id, set())
        return tasks


In [ ]:
from pydantic import BaseModel, field_validator
from typing import List

class Service(BaseModel):
    name: str
    dependencies: List[str] = []
    entrypoint: bool = False

class MicroserviceConfig(BaseModel):
    services: List[Service]

    @field_validator("services")
    @classmethod
    def validate_services(cls, services):
        names = {s.name for s in services}
        for s in services:
            for dep in s.dependencies:
                if dep not in names:
                    raise ValueError(f"Dependency {dep} not found")
        if not any(s.entrypoint for s in services):
            raise ValueError("At least one service must be entrypoint=True")
        # Detect cycles (simplified)
        visited = set()
        def dfs(service, stack):
            if service.name in stack:
                raise ValueError("Cyclic dependency detected")
            stack.add(service.name)
            for dep in service.dependencies:
                dfs(next(s for s in services if s.name == dep), stack)
            stack.remove(service.name)
        for s in services:
            dfs(s, set())
        return services


In [ ]:
from pydantic import BaseModel, field_validator
from typing import List

class Factory(BaseModel):
    name: str
    stock: int

class Warehouse(BaseModel):
    name: str
    stock: int

class RetailStore(BaseModel):
    name: str
    stock: int

class SupplyChain(BaseModel):
    factories: List[Factory]
    warehouses: List[Warehouse]
    stores: List[RetailStore]

    @field_validator("warehouses")
    @classmethod
    def validate_warehouses(cls, warehouses):
        for w in warehouses:
            if w.stock < 0:
                raise ValueError("Warehouse stock cannot be negative")
        return warehouses

    @field_validator("stores")
    @classmethod
    def validate_stores(cls, stores, values):
        for s in stores:
            if s.stock < 0:
                raise ValueError("Store stock cannot be negative")
        # Balance check: total warehouse stock == total store stock
        if "warehouses" in values:
            total_wh = sum(w.stock for w in values["warehouses"])
            total_st = sum(s.stock for s in stores)
            if total_wh != total_st:
                raise ValueError("Warehouse and store stock must balance")
        return stores



---

# 🧩 Topic 3: Advanced Validation & Cross‑Field Rules

## 🎯 Very Easy (2–5 challenges)
1. **RegistrationForm** → `password` and `confirm_password` must match.  
2. **AgeCheck** → `date_of_birth` must make user ≥ 18 years old.  
3. **EmailSignup** → `email` must contain `@` and `domain`.  

---

## 🎯 Easy (2–5 challenges)
1. **Booking** → `end_date` must be after `start_date`.  
2. **Event** → `start_time` must be before `end_time`.  
3. **Subscription** → `trial_end` must not exceed 30 days after `trial_start`.  
4. **Delivery** → `delivery_date` must be ≥ `order_date`.  

---

## 🎯 Medium (2–5 challenges)
1. **Discount** → Only one of `percentage` or `amount` can be set (mutually exclusive).  
2. **Payment** → If `card_number` is provided, `expiry_date` and `cvv` must also be present.  
3. **ProfileUpdate** → If `email` is updated, `email_verified` must reset to False.  
4. **Reservation** → `guest_count` must not exceed `max_capacity`.  

---

## 🎯 Very Medium (2–5 challenges)
1. **EmployeeCompensation** → `bonus` cannot exceed 20% of `salary`.  
2. **CourseEnrollment** → `end_date` must be within `semester_end`.  
3. **VehicleRegistration** → If `vehicle_type="electric"`, `battery_capacity` must be provided.  
4. **Membership** → If `membership_type="premium"`, `payment_info` must be present.  

---

## 🎯 Hard (2–5 challenges)
1. **Flight** → `departure_airport` ≠ `arrival_airport`; `route_code` must match both.  
2. **ConferenceRoomBooking** → Room capacity must be ≥ `attendees`.  
3. **InsurancePolicy** → If `coverage="health"`, `medical_history` must be provided.  
4. **Shipment** → `weight` must not exceed `max_allowed_weight`.  

---

## 🎯 Very Hard (2–5 challenges)
1. **LoanApplication** → Loan amount ≤ 5× income unless collateral is provided.  
2. **JobApplication** → If `experience_years < 2`, `internship_details` must be present.  
3. **TravelVisa** → If `country="USA"`, `passport_validity` must be ≥ 6 months.  
4. **OnlineExam** → `end_time` must be within `exam_window`.  

---

## 🎯 Extreme Hard (2–5 challenges)
1. **ConferenceSchedule** → Sessions cannot overlap in time.  
2. **ProjectTimeline** → Tasks must fit within project start/end dates.  
3. **HospitalShift** → Doctors cannot be assigned overlapping shifts.  
4. **SportsTournament** → Matches must not overlap in the same stadium.  

---

## 🎯 Very Extreme Hard (2–5 challenges)
1. **HealthcareRecord** → Prescriptions must be signed by the doctor.  
2. **InsuranceCoverage** → Insurance must cover prescribed drugs.  
3. **PatientTreatment** → Patient age must match treatment eligibility.  
4. **BankingTransaction** → Transfers must balance debits and credits across accounts.  
5. **SupplyChainAudit** → Shipments must reconcile with warehouse inventory logs.  

---





---

# 🧩 Topic 4: Error Handling & Robustness

## 🎯 Difficulty Ladder

### **Very Easy (2–5 challenges)**
1. **SimpleValidationError** → Catch invalid email format and return a clear error message.  
2. **RequiredFieldCheck** → Ensure required fields (`username`, `password`) are present.  
3. **DefaultFallback** → If `nickname` is missing, default to `username`.  

---

### **Easy (2–5 challenges)**
1. **RetryCounter** → Track number of retries for a failed API call.  
2. **GracefulTimeout** → If `timeout` exceeds threshold, raise a controlled error.  
3. **FallbackValue** → If `currency` is missing, default to `"USD"`.  
4. **ValidationErrorWrapper** → Wrap Pydantic errors into a custom error response.  

---

### **Medium (2–5 challenges)**
1. **PaymentError** → If `amount <= 0`, raise `InvalidPaymentError`.  
2. **WebhookRetry** → Retry webhook validation up to 3 times before failing.  
3. **DatabaseConnection** → If connection string invalid, raise `DatabaseError`.  
4. **GracefulDegradation** → If optional service fails, continue with reduced functionality.  

---

### **Very Medium (2–5 challenges)**
1. **APIResponseValidator** → Ensure API response contains required keys; raise error otherwise.  
2. **CircuitBreaker** → If 5 consecutive failures occur, block further requests for 1 minute.  
3. **ErrorLogging** → Log all validation errors to a file.  
4. **StructuredError** → Return errors in JSON format (`code`, `message`).  

---

### **Hard (2–5 challenges)**
1. **TransactionRollback** → If any step in transaction fails, rollback all changes.  
2. **MultiErrorAggregation** → Collect multiple validation errors and return them together.  
3. **DeadLetterQueue** → Failed messages are sent to a DLQ for later inspection.  
4. **GracefulShutdown** → On fatal error, close resources cleanly.  

---

### **Very Hard (2–5 challenges)**
1. **DistributedRetry** → Retry failed jobs across multiple workers.  
2. **PartialSuccess** → Allow partial batch processing with error reporting.  
3. **CustomExceptionHierarchy** → Define domain‑specific exceptions (`PaymentError`, `AuthError`).  
4. **ErrorMetrics** → Track error counts and expose metrics for monitoring.  

---

### **Extreme Hard (2–5 challenges)**
1. **ResilientPipeline** → Build a pipeline that continues processing even if one stage fails.  
2. **AdaptiveRetry** → Retry with exponential backoff.  
3. **FailoverMechanism** → Switch to backup service if primary fails.  
4. **ErrorCorrelationID** → Attach correlation IDs to errors for tracing.  

---

### **Very Extreme Hard (2–5 challenges)**
1. **GlobalErrorHandler** → Centralized error handling across microservices.  
2. **ChaosTesting** → Inject random failures to test resilience.  
3. **Self‑HealingSystem** → Detect failure and auto‑recover.  
4. **AuditTrail** → Maintain immutable logs of all errors for compliance.  
5. **Cross‑ServiceErrorPropagation** → Ensure errors propagate correctly across service boundaries.  

---


---



---

# 🧩 Topic 5: Serving & API Integration

## 🎯 Difficulty Ladder

### **Very Easy (2–5 challenges)**
1. **Basic FastAPI Endpoint** → `/ping` returns `"pong"`.  
2. **Simple Model Serving** → `/user` accepts a `User` model and returns it back.  
3. **Echo Service** → `/echo` returns the same payload sent.  

---

### **Easy (2–5 challenges)**
1. **Validation Error Response** → `/register` validates `RegistrationForm` and returns structured errors.  
2. **Default Values in API** → `/profile` sets default nickname if missing.  
3. **Basic Query Params** → `/search?query=abc` returns query string.  
4. **Path Params** → `/items/{item_id}` returns item details.  

---

### **Medium (2–5 challenges)**
1. **Nested Models in API** → `/order` accepts `Order` with nested `Product`s.  
2. **Conditional Validation** → `/discount` enforces mutually exclusive fields.  
3. **Custom Error Handling** → Return JSON error with `code` and `message`.  
4. **Response Model** → `/invoice` returns validated `Invoice` response.  

---

### **Very Medium (2–5 challenges)**
1. **Cross‑Field Rules in API** → `/loan` enforces loan rules (≤5× income unless collateral).  
2. **Dependency Injection** → Inject DB connection into endpoints.  
3. **Error Logging Middleware** → Log all request errors.  
4. **Circuit Breaker Middleware** → Block requests after repeated failures.  

---

### **Hard (2–5 challenges)**
1. **Transactional API** → `/transaction` rolls back on failure.  
2. **Batch Processing API** → `/batch` allows partial success with error reporting.  
3. **DLQ Integration** → Failed requests are sent to DLQ.  
4. **Graceful Shutdown Hook** → Close DB connections on shutdown.  

---

### **Very Hard (2–5 challenges)**
1. **Distributed Retry API** → Retry failed jobs across workers.  
2. **Failover API** → Switch to backup service if primary fails.  
3. **Adaptive Retry with Backoff** → Retry with exponential backoff.  
4. **Error Metrics Endpoint** → `/metrics` exposes error counts.  

---

### **Extreme Hard (2–5 challenges)**
1. **Resilient Pipeline API** → `/pipeline` continues processing despite failures.  
2. **Correlation ID Middleware** → Attach correlation IDs to all errors.  
3. **Self‑Healing API** → Detect failure and auto‑recover.  
4. **Audit Trail Endpoint** → `/audit` returns immutable error logs.  

---

### **Very Extreme Hard (2–5 challenges)**
1. **Global Error Handler** → Centralized error handling across all endpoints.  
2. **Chaos Testing Endpoint** → `/chaos` injects random failures.  
3. **Cross‑Service Error Propagation** → Errors propagate across microservices.  
4. **Compliance Logging** → Immutable logs for regulatory audits.  
5. **Enterprise Observability** → Structured logs + metrics + tracing.  

---




---

# 🧩 Topic 6: Advanced Features & Integrations

## 🎯 Difficulty Ladder

### **Very Easy (2–5 challenges)**
1. **Field Metadata** → Use `Field(title, description, example)` for schema docs.  
2. **Strict Mode** → Enforce strict types (`ConfigDict(strict=True)`).  
3. **Extra Fields** → Forbid unknown fields (`extra="forbid"`).  

---

### **Easy (2–5 challenges)**
1. **Serialization** → `.model_dump()` and `.model_dump_json()`.  
2. **Deserialization** → `.model_validate()` from dict/JSON.  
3. **Schema Generation** → `.model_json_schema()` for OpenAPI docs.  
4. **Default Factory** → Use `Field(default_factory=...)`.  

---

### **Medium (2–5 challenges)**
1. **Generic Models** → `PaginatedResponse[T]` with `TypeVar`.  
2. **Custom Constrained Types** → `constr`, `conint`, `conlist`.  
3. **Standard Types** → `EmailStr`, `HttpUrl`, `IPvAnyAddress`.  
4. **Union Types** → Accept multiple input formats.  

---

### **Very Medium (2–5 challenges)**
1. **ORM Mode** → Validate SQLAlchemy objects with `ConfigDict(from_attributes=True)`.  
2. **Aliasing** → Use `Field(alias="external_name")`.  
3. **Computed Fields** → Add derived fields with `@computed_field`.  
4. **Custom Validators** → Reusable validators with `field_validator`.  

---

### **Hard (2–5 challenges)**
1. **Complex Generics** → Nested generics (`Response[List[T]]`).  
2. **Custom Error Messages** → Override default error messages.  
3. **Advanced Serialization** → Custom encoders for datetime, Decimal.  
4. **Immutable Models** → `ConfigDict(frozen=True)`.  

---

### **Very Hard (2–5 challenges)**
1. **Integration with SQLAlchemy** → Map ORM models to Pydantic.  
2. **Integration with FastAPI** → Use response models with schema metadata.  
3. **Integration with Celery/Kafka** → Validate task payloads.  
4. **Integration with Airflow** → Validate DAG configs.  

---

### **Extreme Hard (2–5 challenges)**
1. **Custom Root Types** → Models that wrap lists/dicts.  
2. **Advanced Generics with Constraints** → e.g., `PositiveIntResponse[T]`.  
3. **Schema Customization for OpenAPI** → Add examples, descriptions, deprecations.  
4. **Cross‑Library Integration** → Pydantic with Marshmallow or dataclasses.  

---

### **Very Extreme Hard (2–5 challenges)**
1. **Enterprise Schema Registry** → Generate schemas for Kafka/Avro.  
2. **Dynamic Model Creation** → `create_model()` at runtime.  
3. **Plugin Architecture** → Extend Pydantic with custom plugins.  
4. **Multi‑Service Integration** → Shared models across microservices.  
5. **Compliance‑Ready Models** → GDPR/PCI‑DSS validation rules.  

---

